# Temporal convolutional models on the 8-hour grid

The shared sequence request fixes the TCN architecture, feature order, chronological folds,
missing-observation policy, and full checkpoint schedule before training. Every complete
checkpoint remains a distinct prediction identity for later validation backtests.

**Learning objectives**

- construct a temporal-convolution request on an explicit observation cadence;
- inspect receptive-field, gap-policy, and checkpoint identity; and
- verify exact validation coverage and fitted-state persistence.

**Book reference:** Chapter 19, convolutional sequence models.

**Prerequisites:** finalized crypto labels, features, and purged walk-forward folds; CUDA for the
canonical run.

In [1]:
import os

import polars as pl

from case_studies.crypto_perps_funding.research_workflow import (
    REGRESSION_LABELS,
    declared_contracts,
    freeze_official_model_population,
    model_request_catalog,
    open_study,
    plan_model_catalog,
    plan_specs,
    run_model_plan,
)

In [2]:
EXECUTION_TIER = "canonical"
SUPERSEDES_POPULATION: str = ""
# The generation of this notebook's own checkpoint population that this run replaces, if any.
# Distinct from SUPERSEDES_POPULATION above, which is the case-wide official model population:
# the two are separate declarations and a refit can move either without moving the other.
SUPERSEDES_MODEL_POPULATION: str = ""
WORKSPACE = os.environ.get("ML4T_OUTPUT_DIR", "")
LABELS = REGRESSION_LABELS
PREVIEW_REDUCTIONS = {}
OVERRIDES = {"device": "cuda"}

## Resolve sequence and checkpoint identities

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
official_population = (
    freeze_official_model_population(study, supersedes=SUPERSEDES_POPULATION or None)
    if EXECUTION_TIER == "canonical"
    else None
)
requests = model_request_catalog("deep_learning", labels=LABELS, config_prefix="tcn")
requests

family,label,config_name
str,str,str
"""deep_learning""","""fwd_ret_8h""","""tcn"""


In [4]:
plan = plan_model_catalog(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides=OVERRIDES,
    preview_reductions=PREVIEW_REDUCTIONS,
)
# Sequence eligibility follows from the resolved gap policy and lookback, so read both from the
# frozen specification instead of restating the configuration file here.
resolved_preprocessing = [spec["computation"]["preprocessing"] for spec in plan_specs(plan)]
contracts = declared_contracts(plan).with_columns(
    pl.Series("gap_policy", [step["gap_policy"] for step in resolved_preprocessing]),
    pl.Series("lookback", [step["lookback"] for step in resolved_preprocessing]),
)
contracts.select(
    "label",
    "config_name",
    "gap_policy",
    "lookback",
    "checkpoint_value",
    "eligible_rows",
    "training_hash",
)

label,config_name,gap_policy,lookback,checkpoint_value,eligible_rows,training_hash
str,str,str,i64,i64,i64,str
"""fwd_ret_8h""","""tcn""","""exclude_windows_crossing_missi…",60,5,31885,"""72a628e0ec33"""
"""fwd_ret_8h""","""tcn""","""exclude_windows_crossing_missi…",60,10,31885,"""72a628e0ec33"""
"""fwd_ret_8h""","""tcn""","""exclude_windows_crossing_missi…",60,15,31885,"""72a628e0ec33"""
"""fwd_ret_8h""","""tcn""","""exclude_windows_crossing_missi…",60,20,31885,"""72a628e0ec33"""
"""fwd_ret_8h""","""tcn""","""exclude_windows_crossing_missi…",60,25,31885,"""72a628e0ec33"""
…,…,…,…,…,…,…
"""fwd_ret_8h""","""tcn""","""exclude_windows_crossing_missi…",60,80,31885,"""72a628e0ec33"""
"""fwd_ret_8h""","""tcn""","""exclude_windows_crossing_missi…",60,85,31885,"""72a628e0ec33"""
"""fwd_ret_8h""","""tcn""","""exclude_windows_crossing_missi…",60,90,31885,"""72a628e0ec33"""


The complete case-wide population is recorded before the first fit, so a member that later
fails to train cannot quietly disappear from the population it was declared in. This notebook
produces one slice of it, and that slice must lie inside the declaration.

In [5]:
if official_population is not None:
    outside = set(plan.expected_prediction_hashes) - set(official_population.members)
    if outside:
        raise RuntimeError(
            f"{len(outside)} declared checkpoints lie outside the official model population"
        )

## Execute the declared population

In [6]:
execution = run_model_plan(
    plan,
    supersedes=SUPERSEDES_MODEL_POPULATION or None,
    population_name="crypto-tcn-validation-predictions-v1"
    if EXECUTION_TIER == "canonical"
    else None,
)
catalog = execution.catalog_rows.sort("label", "config_name", "checkpoint_value")
if (
    catalog.height != len(plan.expected_prediction_hashes)
    or catalog.filter(~pl.col("complete")).height
):
    raise RuntimeError("TCN checkpoint population is incomplete")
catalog.select(
    "label",
    "config_name",
    "checkpoint_value",
    "training_hash",
    "prediction_hash",
    "complete",
)

label,config_name,checkpoint_value,training_hash,prediction_hash,complete
str,str,i64,str,str,bool
"""fwd_ret_8h""","""tcn""",5,"""72a628e0ec33""","""0e54627a3c97""",true
"""fwd_ret_8h""","""tcn""",10,"""72a628e0ec33""","""0c9794811179""",true
"""fwd_ret_8h""","""tcn""",15,"""72a628e0ec33""","""8e5683df5143""",true
"""fwd_ret_8h""","""tcn""",20,"""72a628e0ec33""","""e4114dd4215f""",true
"""fwd_ret_8h""","""tcn""",25,"""72a628e0ec33""","""a8ec401d4f64""",true
…,…,…,…,…,…
"""fwd_ret_8h""","""tcn""",80,"""72a628e0ec33""","""6aa246504b06""",true
"""fwd_ret_8h""","""tcn""",85,"""72a628e0ec33""","""f45b6cea7652""",true
"""fwd_ret_8h""","""tcn""",90,"""72a628e0ec33""","""ce90e512121f""",true


## Key takeaways and limitations

- Dilated convolutions use a fixed chronological input window whose eligible keys are known before
  fitting.
- Gap handling and checkpoint membership remain visible in the resolved request.
- The configured receptive field limits the temporal dependencies the model can represent.